# Test Metrics

Consolidated multi-period metrics for regular RCAN checkpoints, period-specific DANN checkpoints, and bilinear baselines.

In [1]:
# =========================================================
# imports
# =========================================================
import math
import os
from contextlib import nullcontext
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from models import ClimateRCAN_ShallowFusion, ClimateRCAN_ShallowFusion_DANN


In [2]:
# =========================================================
# user settings
# =========================================================
DATA_ROOT = "/projects/sds-lab/Shuochen/downscaling/preprocessed"

RCM_VAR = "tas"
GCM_NAME = "CanESM2"
RCM_NAME = "RCA4"
GRID = "NAM-44i"
RCM_PRODUCT = "raw"
FACTOR = 4

# Branch for checkpoint/model metrics. Use "GCM_RCM" or "RCM_RCM".
EXP = "GCM_RCM"

# Branch for the bilinear baseline. Use EXP, "GCM_RCM", or "RCM_RCM".
BILINEAR_EXP = EXP

METRIC_RANGES = [
    (2006, 2099),
    (2006, 2040),
    (2041, 2070),
    (2071, 2099),
]

TRAIN_START_YEAR = 1951
TRAIN_END_YEAR = 2005
VAL_START_YEAR = 2006
VAL_END_YEAR = 2099

INPUT_FILE = "low_res.pth"
TARGET_FILE = "high_res.pth"
HR_MASK_FILE = "high_res_mask.pth"
HR_ELEVATION_FILE = "high_res_elevation.pth"

MODEL_BATCH_SIZE = None  # None uses checkpoint config batch_size, then falls back to 256.
BILINEAR_BATCH_SIZE = 64
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available() if "torch" in globals() else False

# Regular RCAN checkpoint to evaluate over METRIC_RANGES.
REGULAR_VALSET_MODEL_NAME = "RCAN_HR_Aux_ShallowFusion"
REGULAR_VALSET_TRIAL_NUM = "0001"
REGULAR_VALSET_RUN_DIR = "optuna_lr_gcm_with_hr_mask_elevation_to_hr_rcm_mse_shallowfusion"

# DANN checkpoints trained separately for each validation period.
DANN_VALSET_MODEL_NAME = "RCAN_HR_Aux_ShallowFusion_DANN"
DANN_BEST_TRIAL_NUM_BY_PERIOD = {
    (2006, 2040): "0012",
    (2041, 2070): "0092",
    (2071, 2099): "0186",
}
DANN_RUN_DIR_BY_PERIOD = {
    (2006, 2040): "dann_2006_2040_bs512",
    (2041, 2070): "dann_2041_2070_bs512",
    (2071, 2099): "dann_2071_2099_bs512",
}

# Set to one tuple, for example (2006, 2040), to show only that period.
# Leave as None to show the three matching DANN validation periods.
DANN_PERIOD_TO_SHOW = None

VALID_EXPS = {"GCM_RCM", "RCM_RCM"}
if EXP not in VALID_EXPS:
    raise ValueError(f"EXP must be one of {sorted(VALID_EXPS)}, got {EXP!r}")
if BILINEAR_EXP not in VALID_EXPS:
    raise ValueError(f"BILINEAR_EXP must be one of {sorted(VALID_EXPS)}, got {BILINEAR_EXP!r}")


In [3]:
# =========================================================
# shared helpers
# =========================================================
def model_is_dann(model_name):
    return model_name == "RCAN_HR_Aux_ShallowFusion_DANN"


def make_args(exp=EXP):
    return SimpleNamespace(
        rcm_var=RCM_VAR,
        gcm_name=GCM_NAME,
        rcm_name=RCM_NAME,
        grid=GRID,
        rcm_product=RCM_PRODUCT,
        factor=FACTOR,
        exp=exp,
        data_root=DATA_ROOT,
        input_file=INPUT_FILE,
        target_file=TARGET_FILE,
        hr_mask_file=HR_MASK_FILE,
        hr_elevation_file=HR_ELEVATION_FILE,
        train_start_year=TRAIN_START_YEAR,
        train_end_year=TRAIN_END_YEAR,
        val_start_year=VAL_START_YEAR,
        val_end_year=VAL_END_YEAR,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )


def apply_config_to_args(args, config):
    for key in (
        "rcm_var",
        "gcm_name",
        "rcm_name",
        "grid",
        "rcm_product",
        "factor",
        "exp",
        "input_file",
        "target_file",
        "hr_mask_file",
        "hr_elevation_file",
        "train_start_year",
        "train_end_year",
        "val_start_year",
        "val_end_year",
    ):
        if key in config:
            setattr(args, key, config[key])
    return args


def experiment_folder(args):
    return os.path.join(
        args.data_root,
        (
            f"{args.rcm_var}.{args.gcm_name}.{args.rcm_name}.day."
            f"{args.grid}.{args.rcm_product}.{args.exp}"
        ),
    )


def checkpoint_path(model_name, trial_num, exp, run_dir):
    trial_num = f"{int(trial_num):04d}"
    args = make_args(exp=exp)
    return Path(experiment_folder(args)) / "trained_models" / model_name / run_dir / f"trial_{trial_num}" / "best_model.pth"


def split_indices(args, n_samples):
    if args.train_end_year < args.train_start_year:
        raise ValueError("train_end_year must be >= train_start_year.")
    if args.val_end_year < args.val_start_year:
        raise ValueError("val_end_year must be >= val_start_year.")

    if args.gcm_name == "CanESM2":
        train_start_idx = 0
        train_end_idx = (args.train_end_year - args.train_start_year + 1) * 365
        val_start_idx = (args.val_start_year - args.train_start_year) * 365
        val_end_idx = (args.val_end_year - args.train_start_year + 1) * 365
    elif args.gcm_name == "EC-EARTH":
        def is_leap(year):
            return (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0)

        def days_between_years(start_year, end_year_inclusive):
            if end_year_inclusive < start_year:
                return 0
            return sum(
                366 if is_leap(year) else 365
                for year in range(start_year, end_year_inclusive + 1)
            )

        train_start_idx = 0
        train_end_idx = days_between_years(args.train_start_year, args.train_end_year)
        val_start_idx = days_between_years(args.train_start_year, args.val_start_year - 1)
        val_end_idx = days_between_years(args.train_start_year, args.val_end_year)
    else:
        raise ValueError(f"Year split is not defined for gcm_name={args.gcm_name}")

    if train_end_idx > n_samples:
        raise ValueError(f"Training end index {train_end_idx} exceeds {n_samples} samples.")
    if val_end_idx > n_samples:
        raise ValueError(f"Validation end index {val_end_idx} exceeds {n_samples} samples.")
    return train_start_idx, train_end_idx, val_start_idx, val_end_idx


def year_range_indices(args, start_year, end_year):
    range_args = SimpleNamespace(**vars(args))
    range_args.val_start_year = int(start_year)
    range_args.val_end_year = int(end_year)
    _, _, start_idx, end_idx = split_indices(range_args, 10**18)
    return start_idx, end_idx


def match_sample_dim(tensor, n_samples, name):
    if tensor.shape[0] == n_samples:
        return tensor
    if tensor.shape[0] == 1:
        print(f"{name} has one sample; expanding to {n_samples} samples.")
        return tensor.expand(n_samples, -1, -1, -1).contiguous()
    raise ValueError(f"{name} has {tensor.shape[0]} samples, expected {n_samples} or 1.")


def assert_finite(tensor, name):
    if not torch.isfinite(tensor).all():
        raise ValueError(f"{name} contains NaN or Inf values.")


def safe_std(tensor):
    std = tensor.std()
    if std.item() == 0:
        raise ValueError("Cannot normalize with zero standard deviation.")
    return std


def stat_tensor(value, fallback):
    if value is None:
        value = fallback
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().float()
    return torch.tensor(float(value), dtype=torch.float32)


def load_dataset(args, checkpoint=None, include_elevation=True):
    exp_folder = experiment_folder(args)
    X = torch.load(os.path.join(exp_folder, args.input_file)).float()
    y = torch.load(os.path.join(exp_folder, args.target_file)).float()
    mask_hr = torch.load(os.path.join(exp_folder, args.hr_mask_file)).float()

    if X.ndim != 4 or y.ndim != 4 or mask_hr.ndim != 4:
        raise ValueError("Expected X, y, and mask_hr to all have shape [N, C, H, W].")
    if X.shape[1] != 1 or y.shape[1] != 1 or mask_hr.shape[1] != 1:
        raise ValueError("Expected one channel each for LR input, HR target, and HR mask.")
    if X.shape[0] != y.shape[0]:
        raise ValueError(f"Sample mismatch: X has {X.shape[0]}, y has {y.shape[0]}.")

    mask_hr = match_sample_dim(mask_hr, y.shape[0], "mask_hr")
    if mask_hr.shape != y.shape:
        raise ValueError(f"Mask shape mismatch: {mask_hr.shape} vs {y.shape}.")
    if y.shape[-2] != args.factor * X.shape[-2] or y.shape[-1] != args.factor * X.shape[-1]:
        raise ValueError("HR target dimensions do not match factor * LR input dimensions.")

    mask_hr = (mask_hr > 0.5).float().contiguous()

    if include_elevation:
        elev_hr = torch.load(os.path.join(exp_folder, args.hr_elevation_file)).float()
        if elev_hr.ndim != 4 or elev_hr.shape[1] != 1:
            raise ValueError("Expected HR elevation to have shape [N, 1, H, W].")
        elev_hr = match_sample_dim(elev_hr, y.shape[0], "elev_hr")
        if elev_hr.shape != y.shape:
            raise ValueError(f"Elevation shape mismatch: {elev_hr.shape} vs {y.shape}.")
        elev_hr = torch.nan_to_num(elev_hr, nan=0.0, posinf=0.0, neginf=0.0).contiguous()
        hr_aux = torch.cat([mask_hr, elev_hr], dim=1).contiguous()
        assert_finite(elev_hr, "elev_hr")
        assert_finite(hr_aux, "hr_aux")
    else:
        elev_hr = None
        hr_aux = None

    assert_finite(X, "X")
    assert_finite(y, "y")
    assert_finite(mask_hr, "mask_hr")

    train_start_idx, train_end_idx, val_start_idx, val_end_idx = split_indices(args, X.shape[0])
    X_train = X[train_start_idx:train_end_idx]
    y_train = y[train_start_idx:train_end_idx]

    if checkpoint is None:
        X_mean = X_train.mean()
        X_std = safe_std(X_train)
        y_mean = y_train.mean()
        y_std = safe_std(y_train)
    else:
        X_mean = stat_tensor(checkpoint.get("X_mean"), X_train.mean())
        X_std = stat_tensor(checkpoint.get("X_std"), safe_std(X_train))
        y_mean = stat_tensor(checkpoint.get("y_mean"), y_train.mean())
        y_std = stat_tensor(checkpoint.get("y_std"), safe_std(y_train))

    input_label = "LR GCM" if args.exp == "GCM_RCM" else "LR RCM"
    print("Experiment branch:", args.exp)
    print("Experiment folder:", exp_folder)
    print("Input branch:", input_label)
    print("Loaded shapes:")
    print("X      :", X.shape)
    print("hr_mask:", mask_hr.shape)
    print("y      :", y.shape)
    if include_elevation:
        print("hr_aux :", hr_aux.shape)
    print("Training normalization split:")
    print(f"years      : {args.train_start_year}-{args.train_end_year}")
    print("X_train   :", X_train.shape)
    print("y_train   :", y_train.shape)
    print("y_mean    :", y_mean.item())
    print("y_std     :", y_std.item())

    return {
        "exp_folder": exp_folder,
        "X": X,
        "y": y,
        "hr_aux": hr_aux,
        "mask_hr": mask_hr,
        "elev_hr": elev_hr,
        "train_start_idx": train_start_idx,
        "train_end_idx": train_end_idx,
        "val_start_idx": val_start_idx,
        "val_end_idx": val_end_idx,
        "X_mean": X_mean,
        "X_std": X_std,
        "y_mean": y_mean,
        "y_std": y_std,
    }


def build_model_from_checkpoint(checkpoint, model_name):
    config = checkpoint.get("config", {})
    required = ["num_resblk", "num_features", "num_groups", "reduction", "res_scale"]
    missing = [key for key in required if key not in config]
    if missing:
        raise KeyError(f"Checkpoint config is missing required model keys: {missing}")

    common_kwargs = dict(
        num_resblk=config["num_resblk"],
        num_features=config["num_features"],
        input_channels=config.get("input_channels", 1),
        output_channels=1,
        hr_aux_channels=config.get("hr_aux_channels", 2),
        scale=config.get("factor", FACTOR),
        num_groups=config["num_groups"],
        reduction=config["reduction"],
        res_scale=config["res_scale"],
    )

    if model_is_dann(model_name):
        model = ClimateRCAN_ShallowFusion_DANN(
            **common_kwargs,
            domain_hidden_dim=config.get("domain_hidden_dim", 128),
            num_domains=2,
        )
    elif model_name == "RCAN_HR_Aux_ShallowFusion":
        model = ClimateRCAN_ShallowFusion(**common_kwargs)
    else:
        raise ValueError(f"Unsupported checkpoint model_name: {model_name}")

    model.load_state_dict(checkpoint["model_state_dict"])
    return model


def forward_prediction(model, model_name, Xn, aux):
    if model_is_dann(model_name):
        y_pred_n, _ = model(
            Xn,
            aux,
            lambda_grl=0.0,
            predict=True,
            classify_domain=False,
        )
        return y_pred_n
    return model(Xn, aux)


def eval_data_for_year_range(data, base_args, start_year, end_year):
    range_args = SimpleNamespace(**vars(base_args))
    range_args.val_start_year = int(start_year)
    range_args.val_end_year = int(end_year)
    _, _, start_idx, end_idx = split_indices(range_args, data["X"].shape[0])
    if end_idx <= start_idx:
        raise ValueError(f"Empty validation range: {start_year}-{end_year}")

    X_eval = data["X"][start_idx:end_idx].contiguous()
    y_eval = data["y"][start_idx:end_idx].contiguous()
    aux_eval = data["hr_aux"][start_idx:end_idx].contiguous()
    mask_eval = data["mask_hr"][start_idx:end_idx].contiguous()
    X_eval_n = ((X_eval - data["X_mean"]) / data["X_std"]).contiguous()
    y_eval_n = ((y_eval - data["y_mean"]) / data["y_std"]).contiguous()

    return {
        "split_name": f"{start_year}-{end_year}",
        "start_year": int(start_year),
        "end_year": int(end_year),
        "start_idx": start_idx,
        "end_idx": end_idx,
        "X_eval_n": X_eval_n,
        "aux_eval": aux_eval,
        "y_eval_n": y_eval_n,
        "y_eval": y_eval,
        "mask_eval": mask_eval,
    }


def make_eval_loader(eval_data, batch_size):
    eval_set = TensorDataset(
        eval_data["X_eval_n"],
        eval_data["aux_eval"],
        eval_data["y_eval_n"],
        eval_data["y_eval"],
        eval_data["mask_eval"],
    )
    loader_kwargs = {"num_workers": NUM_WORKERS, "pin_memory": PIN_MEMORY}
    if NUM_WORKERS > 0:
        loader_kwargs["persistent_workers"] = True
    return DataLoader(eval_set, batch_size=batch_size, shuffle=False, **loader_kwargs)


def ssim_batch(pred, target, data_range, window_size=11):
    pred = pred.float()
    target = target.float()
    data_range = max(float(data_range), 1e-12)
    c1 = (0.01 * data_range) ** 2
    c2 = (0.03 * data_range) ** 2
    pad = window_size // 2

    pred_pad = F.pad(pred, (pad, pad, pad, pad), mode="reflect")
    target_pad = F.pad(target, (pad, pad, pad, pad), mode="reflect")
    mu_pred = F.avg_pool2d(pred_pad, window_size, stride=1)
    mu_target = F.avg_pool2d(target_pad, window_size, stride=1)

    mu_pred_sq = mu_pred ** 2
    mu_target_sq = mu_target ** 2
    mu_pred_target = mu_pred * mu_target

    sigma_pred_sq = F.avg_pool2d(pred_pad * pred_pad, window_size, stride=1) - mu_pred_sq
    sigma_target_sq = F.avg_pool2d(target_pad * target_pad, window_size, stride=1) - mu_target_sq
    sigma_pred_target = F.avg_pool2d(pred_pad * target_pad, window_size, stride=1) - mu_pred_target

    ssim_map = ((2 * mu_pred_target + c1) * (2 * sigma_pred_target + c2)) / (
        (mu_pred_sq + mu_target_sq + c1) * (sigma_pred_sq + sigma_target_sq + c2)
    )
    return torch.clamp(ssim_map, -1.0, 1.0).mean(dim=(1, 2, 3))


def evaluate_model_period_metrics(model, model_name, eval_loader, eval_data, data, device):
    model.eval()
    loss_fn = nn.MSELoss()
    autocast_ctx = torch.cuda.amp.autocast if device.type == "cuda" else nullcontext
    y_mean_dev = data["y_mean"].to(device)
    y_std_dev = data["y_std"].to(device)
    target_range = float((eval_data["y_eval"].max() - eval_data["y_eval"].min()).item())
    if target_range <= 0:
        target_range = 1.0

    val_loss_sum = 0.0
    val_samples = 0
    full_se_sum = 0.0
    full_pixel_count = 0
    land_se_sum = 0.0
    land_pixel_count = 0.0
    ssim_sum = 0.0
    ssim_count = 0

    with torch.no_grad():
        for Xn, aux, yn, y_raw, mask_raw in eval_loader:
            Xn = Xn.to(device, non_blocking=True)
            aux = aux.to(device, non_blocking=True)
            yn = yn.to(device, non_blocking=True)
            y_raw = y_raw.to(device, non_blocking=True)
            mask_raw = mask_raw.to(device, non_blocking=True)

            with autocast_ctx():
                y_pred_n = forward_prediction(model, model_name, Xn, aux)
                loss = loss_fn(y_pred_n, yn)

            y_pred = y_pred_n * y_std_dev + y_mean_dev
            squared_error = (y_pred - y_raw) ** 2

            batch_size = Xn.shape[0]
            val_loss_sum += loss.item() * batch_size
            val_samples += batch_size
            full_se_sum += squared_error.sum().item()
            full_pixel_count += squared_error.numel()
            land_se_sum += (squared_error * mask_raw).sum().item()
            land_pixel_count += mask_raw.sum().item()

            batch_ssim = ssim_batch(y_pred, y_raw, target_range)
            ssim_sum += batch_ssim.sum().item()
            ssim_count += batch_ssim.numel()

    full_mse = full_se_sum / full_pixel_count
    land_mse = land_se_sum / land_pixel_count
    return {
        "normalized_mse": val_loss_sum / val_samples,
        "physical_mse_full_image": full_mse,
        "physical_mse_land_only": land_mse,
        "full_rmse": math.sqrt(full_mse),
        "land_rmse": math.sqrt(land_mse),
        "psnr": float("inf") if full_mse == 0 else 10.0 * math.log10((target_range ** 2) / full_mse),
        "ssim": ssim_sum / ssim_count,
    }


def data_with_checkpoint_stats(base_data, ckpt):
    ckpt_data = dict(base_data)
    ckpt_data["X_mean"] = stat_tensor(ckpt.get("X_mean"), ckpt_data["X_mean"])
    ckpt_data["X_std"] = stat_tensor(ckpt.get("X_std"), ckpt_data["X_std"])
    ckpt_data["y_mean"] = stat_tensor(ckpt.get("y_mean"), ckpt_data["y_mean"])
    ckpt_data["y_std"] = stat_tensor(ckpt.get("y_std"), ckpt_data["y_std"])
    return ckpt_data


def print_metric_table(rows, include_trial=False):
    if include_trial:
        print(
            f"{'years':<11} {'trial':>6} {'samples':>8} {'normalized_mse':>15} "
            f"{'full_mse':>10} {'land_mse':>10} {'full_rmse':>10} "
            f"{'land_rmse':>10} {'PSNR':>10} {'SSIM':>10}"
        )
    else:
        print(
            f"{'years':<11} {'samples':>8} {'normalized_mse':>15} "
            f"{'full_mse':>10} {'land_mse':>10} {'full_rmse':>10} "
            f"{'land_rmse':>10} {'PSNR':>10} {'SSIM':>10}"
        )

    for row in rows:
        prefix = f"{row['years']:<11} "
        if include_trial:
            prefix += f"{row['trial_num']:>6} "
        print(
            prefix
            + f"{row['samples']:>8d} "
            + f"{row['normalized_mse']:>15.7f} "
            + f"{row['physical_mse_full_image']:>10.3f} "
            + f"{row['physical_mse_land_only']:>10.3f} "
            + f"{row['full_rmse']:>10.3f} "
            + f"{row['land_rmse']:>10.3f} "
            + f"{row['psnr']:>10.3f} "
            + f"{row['ssim']:>10.5f}"
        )


## Regular Model Period Metrics

In [4]:
# =========================================================
# regular model metrics over multiple validation periods
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
regular_ckpt_path = checkpoint_path(
    REGULAR_VALSET_MODEL_NAME,
    REGULAR_VALSET_TRIAL_NUM,
    EXP,
    REGULAR_VALSET_RUN_DIR,
)
if not regular_ckpt_path.exists():
    raise FileNotFoundError(f"Checkpoint does not exist: {regular_ckpt_path}")

regular_ckpt = torch.load(regular_ckpt_path, map_location="cpu")
regular_config = regular_ckpt.get("config", {})
regular_args = apply_config_to_args(make_args(exp=EXP), regular_config)
for key in ("input_file", "target_file", "hr_mask_file", "hr_elevation_file"):
    if key in regular_ckpt:
        setattr(regular_args, key, regular_ckpt[key])

regular_data = load_dataset(regular_args, regular_ckpt, include_elevation=True)
regular_model = build_model_from_checkpoint(regular_ckpt, REGULAR_VALSET_MODEL_NAME).to(device)
regular_model.eval()
regular_batch_size = int(MODEL_BATCH_SIZE or regular_config.get("batch_size", 256))

regular_valset_results = []
for start_year, end_year in METRIC_RANGES:
    period_eval_data = eval_data_for_year_range(regular_data, regular_args, start_year, end_year)
    period_eval_loader = make_eval_loader(period_eval_data, regular_batch_size)
    period_metrics = evaluate_model_period_metrics(
        regular_model,
        REGULAR_VALSET_MODEL_NAME,
        period_eval_loader,
        period_eval_data,
        regular_data,
        device,
    )
    regular_valset_results.append(
        {
            "years": period_eval_data["split_name"],
            "samples": period_eval_data["end_idx"] - period_eval_data["start_idx"],
            **period_metrics,
        }
    )

print("Regular checkpoint:", regular_ckpt_path)
print_metric_table(regular_valset_results)


Experiment branch: GCM_RCM
Experiment folder: /projects/sds-lab/Shuochen/downscaling/preprocessed/tas.CanESM2.RCA4.day.NAM-44i.raw.GCM_RCM
Input branch: LR GCM
Loaded shapes:
X      : torch.Size([54750, 1, 13, 29])
hr_mask: torch.Size([54750, 1, 52, 116])
y      : torch.Size([54750, 1, 52, 116])
hr_aux : torch.Size([54750, 2, 52, 116])
Training normalization split:
years      : 1951-2005
X_train   : torch.Size([20075, 1, 13, 29])
y_train   : torch.Size([20075, 1, 52, 116])
y_mean    : 5.909543037414551
y_std     : 10.123307228088379


KeyboardInterrupt: 

## DANN Period Metrics

In [ ]:
# =========================================================
# DANN metrics for separately trained validation periods
# =========================================================
def selected_dann_periods():
    periods = list(DANN_RUN_DIR_BY_PERIOD)
    if DANN_PERIOD_TO_SHOW is None:
        return periods

    requested = tuple(DANN_PERIOD_TO_SHOW)
    if requested not in DANN_RUN_DIR_BY_PERIOD:
        raise ValueError(f"DANN_PERIOD_TO_SHOW must be one of {periods}, got {requested}.")
    return [requested]


def dann_valset_checkpoint_path(period):
    run_dir = DANN_RUN_DIR_BY_PERIOD[period]
    trial_num = f"{int(DANN_BEST_TRIAL_NUM_BY_PERIOD[period]):04d}"
    args = make_args(exp=EXP)
    return (
        Path(experiment_folder(args))
        / "trained_models"
        / DANN_VALSET_MODEL_NAME
        / run_dir
        / f"trial_{trial_num}"
        / "best_model.pth"
    )


def load_dann_valset_model(period, device):
    ckpt_path = dann_valset_checkpoint_path(period)
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Checkpoint does not exist: {ckpt_path}")

    ckpt = torch.load(ckpt_path, map_location="cpu")
    ckpt_model_name = ckpt.get("model_name", ckpt.get("config", {}).get("model_name", DANN_VALSET_MODEL_NAME))
    if ckpt_model_name != DANN_VALSET_MODEL_NAME:
        raise ValueError(f"Expected {DANN_VALSET_MODEL_NAME}, but checkpoint model_name is {ckpt_model_name}.")

    model = build_model_from_checkpoint(ckpt, DANN_VALSET_MODEL_NAME).to(device)
    model.eval()
    return model, ckpt_path, ckpt


def dann_args_from_checkpoint(ckpt):
    ckpt_config = ckpt.get("config", {})
    ckpt_args = apply_config_to_args(make_args(exp=EXP), ckpt_config)
    for key in ("input_file", "target_file", "hr_mask_file", "hr_elevation_file"):
        if key in ckpt:
            setattr(ckpt_args, key, ckpt[key])
    return ckpt_args, ckpt_config


dann_valset_results = []
dann_base_data_by_exp = {}

for period in selected_dann_periods():
    start_year, end_year = period
    dann_model, dann_ckpt_path, dann_ckpt = load_dann_valset_model(period, device)
    dann_args, dann_config = dann_args_from_checkpoint(dann_ckpt)

    data_key = (dann_args.exp, dann_args.input_file, dann_args.target_file, dann_args.hr_mask_file, dann_args.hr_elevation_file)
    if data_key not in dann_base_data_by_exp:
        dann_base_data_by_exp[data_key] = load_dataset(dann_args, dann_ckpt, include_elevation=True)
    dann_period_data = data_with_checkpoint_stats(dann_base_data_by_exp[data_key], dann_ckpt)
    dann_batch_size = int(MODEL_BATCH_SIZE or dann_config.get("batch_size", 256))

    period_eval_data = eval_data_for_year_range(dann_period_data, dann_args, start_year, end_year)
    period_eval_loader = make_eval_loader(period_eval_data, dann_batch_size)
    period_metrics = evaluate_model_period_metrics(
        dann_model,
        DANN_VALSET_MODEL_NAME,
        period_eval_loader,
        period_eval_data,
        dann_period_data,
        device,
    )

    trial_num = f"{int(DANN_BEST_TRIAL_NUM_BY_PERIOD[period]):04d}"
    dann_valset_results.append(
        {
            "years": period_eval_data["split_name"],
            "run_dir": DANN_RUN_DIR_BY_PERIOD[period],
            "trial_num": trial_num,
            "checkpoint": str(dann_ckpt_path),
            "samples": period_eval_data["end_idx"] - period_eval_data["start_idx"],
            **period_metrics,
        }
    )

print("DANN validation-period checkpoints:")
for row in dann_valset_results:
    print(f"{row['years']}: {row['run_dir']}, trial {row['trial_num']}")
    print(f"  {row['checkpoint']}")
print()
print_metric_table(dann_valset_results, include_trial=True)


## Bilinear Period Metrics

In [ ]:
# =========================================================
# bilinear baseline over multiple validation periods
# =========================================================
def period_tensors(data, args, start_year, end_year):
    start_idx, end_idx = year_range_indices(args, start_year, end_year)
    if end_idx > data["X"].shape[0]:
        raise ValueError(
            f"Range {start_year}-{end_year} ends at index {end_idx}, "
            f"but data has only {data['X'].shape[0]} samples."
        )
    if end_idx <= start_idx:
        raise ValueError(f"Validation range is empty: {start_year}-{end_year}")

    return (
        data["X"][start_idx:end_idx].contiguous(),
        data["y"][start_idx:end_idx].contiguous(),
        data["mask_hr"][start_idx:end_idx].contiguous(),
    )


def evaluate_bilinear_period(data, args, start_year, end_year):
    X_period, y_period, mask_period = period_tensors(data, args, start_year, end_year)
    loader = DataLoader(
        TensorDataset(X_period, y_period, mask_period),
        batch_size=BILINEAR_BATCH_SIZE,
        shuffle=False,
    )

    target_range = float((y_period.max() - y_period.min()).item())
    if target_range <= 0:
        target_range = 1.0

    y_mean = data["y_mean"]
    y_std = data["y_std"]
    normalized_sse = 0.0
    normalized_count = 0
    full_sse = 0.0
    full_count = 0
    land_sse = 0.0
    land_count = 0.0
    ssim_sum = 0.0
    ssim_count = 0

    with torch.no_grad():
        for X_batch, y_batch, mask_batch in loader:
            pred_batch = F.interpolate(
                X_batch,
                size=y_batch.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )

            se = (pred_batch - y_batch) ** 2
            se_n = ((pred_batch - y_mean) / y_std - (y_batch - y_mean) / y_std) ** 2

            normalized_sse += se_n.sum().item()
            normalized_count += se_n.numel()
            full_sse += se.sum().item()
            full_count += se.numel()
            land_sse += (se * mask_batch).sum().item()
            land_count += mask_batch.sum().item()

            batch_ssim = ssim_batch(pred_batch, y_batch, target_range)
            ssim_sum += batch_ssim.sum().item()
            ssim_count += batch_ssim.numel()

    if full_count <= 0:
        raise ValueError(f"Range {start_year}-{end_year} has no pixels.")
    if land_count <= 0:
        raise ValueError(f"Range {start_year}-{end_year} has no land pixels.")

    full_mse = full_sse / full_count
    land_mse = land_sse / land_count
    return {
        "years": f"{start_year}-{end_year}",
        "samples": X_period.shape[0],
        "normalized_mse": normalized_sse / normalized_count,
        "physical_mse_full_image": full_mse,
        "physical_mse_land_only": land_mse,
        "full_rmse": math.sqrt(full_mse),
        "land_rmse": math.sqrt(land_mse),
        "psnr": float("inf") if full_mse == 0 else 10.0 * math.log10((target_range ** 2) / full_mse),
        "ssim": ssim_sum / ssim_count,
    }


bilinear_args = make_args(exp=BILINEAR_EXP)
bilinear_data = load_dataset(bilinear_args, checkpoint=None, include_elevation=False)
bilinear_results = [
    evaluate_bilinear_period(bilinear_data, bilinear_args, start_year, end_year)
    for start_year, end_year in METRIC_RANGES
]

input_label = "LR GCM" if BILINEAR_EXP == "GCM_RCM" else "LR RCM"
print("")
print(f"Bilinear low_res ({input_label}) -> high_res validation baseline [{BILINEAR_EXP}]")
print_metric_table(bilinear_results)
